# RAG Pipeline — Study Notes Assistant (Extended Track)

This notebook builds and evaluates the full RAG pipeline for a **study-notes domain**:
clean & chunk documents → embed → store in a vector DB → retrieve → prompt a local
Ollama LLM → evaluate. It also includes the **Extended Track** Computer Vision component
(YOLO) run over a small image dataset of scanned notes pages / diagrams.

**Domain:** Study notes (lecture PDFs, summary sheets, handwritten-diagram scans)


In [2]:
# Core imports
import os, glob, json, textwrap
from pathlib import Path

import pandas as pd
import numpy as np
from pypdf import PdfReader

import chromadb
from chromadb.utils import embedding_functions

import ollama

DATA_DIR = Path("../data/raw_docs")
IMAGE_DIR = Path("../data/images")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
OLLAMA_MODEL = "llama3.2"
COLLECTION_NAME = "study_notes"
CHUNK_SIZE = 500       # characters
CHUNK_OVERLAP = 75     # characters


## 2.1 Load & Inspect

Load every PDF/text file from `data/raw_docs/`, extract text, and report:
how many documents/pages, which formats, and which files failed to parse
(likely scanned images needing OCR).


In [3]:
def load_documents(data_dir: Path):
    records = []
    failed = []
    for path in sorted(data_dir.glob("**/*")):
        if path.suffix.lower() == ".pdf":
            try:
                reader = PdfReader(str(path))
                text = "\n".join((page.extract_text() or "") for page in reader.pages)
                if len(text.strip()) < 20:
                    failed.append((path.name, "no extractable text (likely scanned - needs OCR)"))
                    continue
                records.append({"source": path.name, "pages": len(reader.pages), "text": text})
            except Exception as e:
                failed.append((path.name, str(e)))
        elif path.suffix.lower() in (".txt", ".md"):
            text = path.read_text(errors="ignore")
            records.append({"source": path.name, "pages": 1, "text": text})
    return records, failed

documents, failed_files = load_documents(DATA_DIR)

print(f"Loaded {len(documents)} documents.")
print(f"Total pages: {sum(d['pages'] for d in documents)}")
print(f"Failed/needs-OCR files: {len(failed_files)}")
for name, reason in failed_files:
    print(f"  - {name}: {reason}")


Loaded 5 documents.
Total pages: 446
Failed/needs-OCR files: 1
  - Introduction to Deep Learning - Day 2 (1).pdf: no extractable text (likely scanned - needs OCR)


**Notes (fill in after running the cell above on your own data):**
- Number of documents / pages: _TODO_
- Formats present: PDF / TXT / MD (update to match your corpus)
- Files that failed to parse or need OCR: _TODO — list them and explain how you handled each_


## 2.2 Chunking Strategy

We use a **fixed-size chunking strategy with overlap** (`CHUNK_SIZE=500` characters,
`CHUNK_OVERLAP=75` characters).

**Justification:** study notes are short, dense paragraphs (definitions, bullet points)
rather than long flowing prose, so a moderate fixed size keeps each chunk focused on
one idea while the overlap preserves context that would otherwise be cut across a
chunk boundary (e.g. a definition that continues on the next line). If your corpus
instead has clear section headers (e.g. numbered topics), consider switching to a
section-based split — the function below is easy to swap out.


In [4]:
def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []  # list of dicts: {id, text, source}
chunk_counter = 0
for doc in documents:
    for c in chunk_text(doc["text"]):
        all_chunks.append({
            "id": f"chunk_{chunk_counter}",
            "text": c,
            "source": doc["source"],
        })
        chunk_counter += 1

print(f"Total chunks created: {len(all_chunks)}")
if all_chunks:
    print("Example chunk:\n", textwrap.shorten(all_chunks[0]["text"], 200))


Total chunks created: 326
Example chunk:
 COMPUTER VISION Part 2 ACTIVATIONS (BASIC PROPERTIES) • Activations are functions that take any real number as input, also known as its domain, and outputs a number in a certain range using a [...]


## 2.3 Embeddings & Vector Store

Generate embeddings for every chunk with a Sentence-Transformers model, store them in
a **Chroma** vector database, and **persist to disk** so the FastAPI backend can load
it directly without rebuilding.


In [5]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Fresh collection each time the notebook is run end-to-end
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

if all_chunks:
    collection.add(
        ids=[c["id"] for c in all_chunks],
        documents=[c["text"] for c in all_chunks],
        metadatas=[{"source": c["source"]} for c in all_chunks],
    )

print(f"Vector store persisted at: {VECTOR_STORE_DIR.resolve()}")
print(f"Collection count: {collection.count()}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store persisted at: C:\Users\MSi\Downloads\rag-assistant-project\rag-assistant-project\backend\data\vector_store
Collection count: 326


## 2.4 Retrieval & Prompting

Implement a retrieval function, test it on **at least 10 sample questions**, and build
the prompt template that combines retrieved context with the question. Citations are
added inline as `[Source: <document>]`.


In [6]:
def retrieve(question: str, top_k: int = 4):
    results = collection.query(query_texts=[question], n_results=top_k)
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    return [{"text": d, "source": (m or {}).get("source", "unknown")} for d, m in zip(docs, metas)]

PROMPT_TEMPLATE = """You are a study-notes assistant. Answer the question using ONLY the
context below. If the answer is not contained in the context, say you don't know.
Cite each fact inline like [Source: <document>].

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question: str, chunks):
    context = "\n\n---\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)


In [7]:
# TODO: replace with 10 real questions about YOUR study notes corpus
sample_questions = [
    "What is <topic 1>?",
    "Explain <topic 2> in simple terms.",
    "What are the main differences between <A> and <B>?",
    "Give an example of <topic 3>.",
    "Summarize the section about <topic 4>.",
    "What formula is used for <topic 5>?",
    "List the steps of <process 1>.",
    "What does <term> mean?",
    "Why is <concept> important?",
    "What are the limitations of <method>?",
]

for q in sample_questions[:2]:
    chunks = retrieve(q)
    print("Q:", q)
    for c in chunks:
        print("  -", c['source'], ":", textwrap.shorten(c['text'], 100))
    print()


Q: What is <topic 1>?
  - MLII.pdf : processing or vector quantization ■ Collaborative filtering, recommendation systems, or [...]
  - MLII.pdf : e) – Multi-modal (often combined with text data) ■ Time-series data: Sensor data, stock [...]
  - MLII.pdf : beled docs SPORTS WORLD NEWS SCIENCE ENTERTAINMENT https://www.coursera.org/learn/ml- [...]
  - Machine Learning D1 S1.pdf : ndary to classify new example. Classification • Every axis(dimension) is a feature • in case [...]

Q: Explain <topic 2> in simple terms.
  - ml_dy3 .pdf : c for different machine learning problems. 1. What is Model Evaluation? Model Evaluation is [...]
  - Machine Learning D1 S1.pdf : ndary to classify new example. Classification • Every axis(dimension) is a feature • in case [...]
  - Machine Learning D1 S1.pdf : to learn through actions. A reward signal indicates if the action is good or bad. • By [...]
  - MLII.pdf : it into Clusters But mind the difference: Clustering Classification • Data is not labeled • 

## 2.5 Vision Component (Extended Track)

Run inference with a pretrained **YOLO** model on the study-notes image dataset
(scanned pages, hand-drawn diagrams, tables). The detected element types (e.g.
`diagram`, `table`, `handwritten-text`) are fed into the RAG prompt as extra context,
so the assistant can say things like *"this page contains a labeled diagram of..."*.

For a first pass, a pretrained general model (e.g. `yolov8n.pt`) is used purely to
demonstrate the pipeline; fine-tuning on your own labeled `diagram/table/figure`
classes will give far more useful detections for this domain — swap the model path
below once you have a fine-tuned checkpoint.


In [9]:
%pip install ultralytics

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.4 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.8 MB/s  0:00:00
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/44.0 MB 3.2 MB/s eta 0:00:14
   - -------------------------------------- 1.3/44.0 MB 3.3 MB/s eta 0:00:14
   - -------------------------------------- 1.6/44.0 MB 3.1 MB/s eta 0:00:14
   -- ------------------------------------- 2.4/44.0 MB 3.0 MB/s eta 0:00:14
   -- ------------------------------------- 2.9/44.0 MB 2.7 MB/s eta 0:00:16
   -- ------------------------------------- 3.1/44.0 MB 2.5 MB/s eta 0:00:17
   --- ------------------------------------ 3.7/44.0 MB 2

In [10]:
from ultralytics import YOLO

# Swap this for your fine-tuned checkpoint once trained, e.g. "runs/detect/train/weights/best.pt"
yolo_model = YOLO("yolov8n.pt")

image_paths = sorted(glob.glob(str(IMAGE_DIR / "*")))
print(f"Found {len(image_paths)} images in {IMAGE_DIR}")

vision_results = {}
for img_path in image_paths[:5]:
    results = yolo_model.predict(img_path, verbose=False)
    labels = []
    for r in results:
        for box in r.boxes:
            labels.append(r.names[int(box.cls[0])])
    vision_results[Path(img_path).name] = labels
    print(Path(img_path).name, "->", labels)


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\MSi\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Found 11 images in ..\data\images
arxiv_2305_06456_4.png -> ['book']
arxiv_2305_06456_7.png -> ['book']
arxiv_2305_06456_9.png -> []
arxiv_2305_08275_7.png -> ['book']
arxiv_2305_08891_0.png -> ['person', 'book', 'person']


**How detections feed into the RAG context:** when a user uploads a page image
alongside their question (see `backend/app/services/vision.py` and the
`/query-image` endpoint), the detected labels are appended to the prompt as an
`[Image detections]` block, so the LLM can ground its answer in both the retrieved
text chunks *and* what was visually detected on the page.


## 2.6 Evaluation

Evaluate the pipeline on **at least 10 test questions**: was the retrieved context
relevant? Was the answer grounded or hallucinated? Results are logged in a table
below.


In [11]:
def ask(question: str, top_k: int = 4):
    chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, chunks)
    response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"], chunks

eval_rows = []
for q in sample_questions:
    try:
        answer, chunks = ask(q)
        retrieved_src = ", ".join(sorted({c['source'] for c in chunks})) if chunks else "none"
    except Exception as e:
        answer, retrieved_src = f"ERROR: {e}", "n/a"
    eval_rows.append({
        "question": q,
        "retrieved_source": retrieved_src,
        "answer": textwrap.shorten(answer, 200),
        "correct": None,  # TODO: fill in True/False after manually checking each answer
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


,question,retrieved_source,answer,correct
0,What is <topic 1>?,n/a,ERROR: model 'llama3.2' not found (status code...,None
1,Explain <topic 2> in simple terms.,n/a,ERROR: model 'llama3.2' not found (status code...,None
2,What are the main differences between <A> and ...,n/a,ERROR: model 'llama3.2' not found (status code...,None
3,Give an example of <topic 3>.,n/a,ERROR: model 'llama3.2' not found (status code...,None
4,Summarize the section about <topic 4>.,n/a,ERROR: model 'llama3.2' not found (status code...,None
5,What formula is used for <topic 5>?,n/a,ERROR: model 'llama3.2' not found (status code...,None
6,List the steps of <process 1>.,n/a,ERROR: model 'llama3.2' not found (status code...,None
7,What does <term> mean?,n/a,ERROR: model 'llama3.2' not found (status code...,None
8,Why is <concept> important?,n/a,ERROR: model 'llama3.2' not found (status code...,None
9,What are the limitations of <method>?,n/a,ERROR: model 'llama3.2' not found (status code...,None


**Failure analysis (fill in after reviewing the table above):**

- Main failure cases observed: _TODO — e.g. retrieval returning a semantically similar
  but factually irrelevant chunk; answers drifting into the LLM's own knowledge when
  the context is thin._
- Mitigations tried: _TODO — e.g. increasing `top_k`, tightening the prompt's
  "answer only from context" instruction, reducing chunk size to increase retrieval
  precision._


## 2.7 Export

The vector store was already persisted to `backend/data/vector_store/` in section 2.3
via `chromadb.PersistentClient`, so the backend loads it directly at startup with no
rebuilding. We also save the pipeline config alongside it for reproducibility.


In [12]:
config = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "collection_name": COLLECTION_NAME,
    "ollama_model": OLLAMA_MODEL,
}

with open(VECTOR_STORE_DIR / "pipeline_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Exported config:")
print(json.dumps(config, indent=2))
print(f"\nVector store + config ready at: {VECTOR_STORE_DIR.resolve()}")
print("Copy/point backend/data/vector_store to this folder before running the API.")


Exported config:
{
  "embedding_model_name": "all-MiniLM-L6-v2",
  "chunk_size": 500,
  "chunk_overlap": 75,
  "collection_name": "study_notes",
  "ollama_model": "llama3.2"
}

Vector store + config ready at: C:\Users\MSi\Downloads\rag-assistant-project\rag-assistant-project\backend\data\vector_store
Copy/point backend/data/vector_store to this folder before running the API.
